Fundamentals of Deep Learning Models

# Lab 03-2: Simple Neural Network
## Exercise: Predicting Iris species with a two-layer neural network

This exercise implements a **two-layer neural network** (Sections 3.6–3.7) with a sigmoid hidden layer and a softmax output layer. The network is trained via full backpropagation using the gradient propagation formula (Eq. 3.26) and the softmax–cross-entropy gradient $\partial J / \partial \mathbf{z}^{[L]} = \hat{\mathbf{y}} - \mathbf{y}$ (Eq. 3.23).
The model predicts iris species (setosa, versicolor, virginica) from four input features.

### Prepare Iris dataset

The Iris dataset contains $N = 150$ samples with $M = 4$ input features and $K = 3$ class labels.

In [1]:
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version

print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)
print('scikit-learn version:', sklearn_version)

NumPy version: 2.0.2
Pandas version: 2.2.2
scikit-learn version: 1.6.1


In [2]:
from sklearn.datasets import load_iris

iris = load_iris()

# iris.data: four features (sepal length/width, petal length/width)
# iris.target: species labels (0=setosa, 1=versicolor, 2=virginica)
iris_df = pd.DataFrame(data= iris.data, columns= iris.feature_names)
iris_tf = pd.DataFrame(data= iris.target, columns= ['species'])

vX = iris_df.copy()
vY = iris_tf['species']

# Change dataset from pandas to numpy
vX = vX.to_numpy()
vY = vY.to_numpy()

### Presenting dataset samples

In [3]:
iris_df = pd.concat([iris_df, iris_tf], axis= 1)
iris_df.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [4]:
print(vY)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


### Splitting data for training and testing

The dataset is split into training and test sets (80/20). Integer class labels are converted to one-hot encoded vectors $\mathbf{y} \in \{0,1\}^K$ for the cross-entropy loss (Eq. 3.16).

In [5]:
from sklearn.model_selection import train_test_split

# Split dataset into 80% train and 20% test
X_train, X_test, y_train_num, y_test = train_test_split(vX, vY, test_size= 0.20, random_state= 101)

# Convert integer labels to one-hot encoding for cross-entropy loss
n_train = y_train_num.shape[0]
y_train = np.zeros((n_train, 3))
for i in range(n_train):
    y_train[i, y_train_num[i]] = 1

### Simple neural network

#### Dense layer: linear transformation

Each layer computes a linear transformation (Section 3.6, forward propagation):

$$\mathbf{z}^{[l]} = \mathbf{W}^{[l]}\mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$

The parameter gradients (Eq. 3.27) for a single sample are:

$$\frac{\partial J}{\partial \mathbf{W}^{[l]}} = \frac{\partial J}{\partial \mathbf{z}^{[l]}} \cdot (\mathbf{a}^{[l-1]})^T, \qquad \frac{\partial J}{\partial \mathbf{b}^{[l]}} = \frac{\partial J}{\partial \mathbf{z}^{[l]}}$$

For a batch of $N$ samples (Eq. 3.39):

$$\frac{\partial J}{\partial \mathbf{W}^{[l]}} = \frac{1}{N} \sum_{i=1}^{N} \frac{\partial J}{\partial \mathbf{z}^{[l],(i)}} \cdot (\mathbf{a}^{[l-1],(i)})^T, \qquad \frac{\partial J}{\partial \mathbf{b}^{[l]}} = \frac{1}{N} \sum_{i=1}^{N} \frac{\partial J}{\partial \mathbf{z}^{[l],(i)}}$$

**Note:** $\partial J / \partial \mathbf{W}^{[l]}$ should have the same shape as $\mathbf{W}^{[l]}$. You must transpose dimensions appropriately.

The fully connected neural network layer is called a Dense layer (or Linear layer), which computes the linear transformation $\mathbf{W}\mathbf{x} + \mathbf{b}$.

In [ ]:
class myDenseLayer:
    def __init__(self, n_out, n_in):
        self.weight = np.empty((n_out, n_in))   # (o, i)
        self.bias = np.zeros((n_out))
        self.saved_x = None     # store x for backpropagation

    def forward(self, x):   # x: (b, i)

        ### START CODE HERE ###

        # Store input for use in backpropagation (Section 3.6)
        self.saved_x = None
        # Compute linear combination: z = W * x^T + b  (Eq. 3.27, forward)
        x_lin = None

        ### END CODE HERE ###

        return x_lin

    def backward(self, x):  # x = dJ/dz (b, o)

        ### START CODE HERE ###

        # Compute weight gradient: dJ/dW = (1/N) * (dJ/dz)^T * a^[l-1]  (Eq. 3.39)
        dw = None
        # Compute bias gradient: dJ/db = (1/N) * sum(dJ/dz)  (Eq. 3.27)
        db = None
        # Propagate gradient to lower layer: dJ/dz * W  (used in Eq. 3.26)
        wdJdz = None

        ### END CODE HERE ###
        
        return dw, db, wdJdz


#### Test Dense layer

In [ ]:
tlyr = myDenseLayer(2,3)
tlyr.weight = np.arange(2,8).reshape(2, 3).astype(np.float32)
tlyr.bias = np.arange(2,4).astype(np.float32)

c = tlyr.forward(np.array([[1.,2.,3.], [3.,4.,5.]]))
d, e, f = tlyr.backward(np.array([[1.,2.], [3.,4.]]))
print('c = [', c[0], c[1], ']\nd = [', d[0], d[1], ']\ne = ', e, '\nf = [', f[0], f[1], ']')


c = [ [22. 41.] [40. 77.] ]
d = [ [5. 7. 9.] [ 7. 10. 13.] ]
e =  [2. 3.] 
f = [ [12. 15. 18.] [26. 33. 40.] ]


**Expected output:**
```
c = [ [22. 41.] [40. 77.] ]
d = [ [5. 7. 9.] [ 7. 10. 13.] ]
e =  [2. 3.]
f = [ [12. 15. 18.] [26. 33. 40.] ]
```

In [8]:
np.random.seed(1)
tlyr = myDenseLayer(2,3)
tlyr.weight = np.random.rand(2,3)
tlyr.bias = np.random.rand(2)

x = np.random.randn(4,3)
c = tlyr.forward(x)
d, e, f = tlyr.backward(np.random.randn(4,2))
print(c)
print(d, '\n'+str(e), '\n'+str(f))

c_exp = [[-1.1105009, 0.43055075], [0.09860334, 0.13921744], [-0.6880153, 0.45549424], [0.84265719, 0.29427352]]
d_exp = [[-0.53420949, 0.9249861, -0.23704371], [0.06934404, 0.10444669, -0.64970057]]
e_exp = [-0.08231073, 0.33804371]
f_exp = [[-0.33731155, -0.25303516, -0.08107993], [0.19380808, 0.11593916, 0.05382117], [-0.11289515, -0.624808, 0.1055763 ], [0.52790358, 0.72318188, 0.04650274]]

print('forward passed.') if np.allclose(c, c_exp) else print('forward failed.')
if not np.allclose(d, d_exp): print('dw failed')
elif not np.allclose(e, e_exp): print('db failed')
elif not np.allclose(f, f_exp): print('wdJdz failed')
else: print('backward passed')

[[-1.1105009   0.43055075]
 [ 0.09860334  0.13921744]
 [-0.6880153   0.45549424]
 [ 0.84265719  0.29427352]]
[[-0.53420949  0.9249861  -0.23704371]
 [ 0.06934404  0.10444669 -0.64970057]] 
[-0.08231073  0.33804371] 
[[-0.33731155 -0.25303516 -0.08107993]
 [ 0.19380808  0.11593916  0.05382117]
 [-0.11289515 -0.624808    0.1055763 ]
 [ 0.52790358  0.72318188  0.04650274]]
forward passed.
backward passed


**Expected output:**
```
[[-1.1105009   0.43055075]
 [ 0.09860334  0.13921744]
 [-0.6880153   0.45549424]
 [ 0.84265719  0.29427352]]
[[-0.53420949  0.9249861  -0.23704371]
 [ 0.06934404  0.10444669 -0.64970057]]
[-0.08231073  0.33804371]
[[-0.33731155 -0.25303516 -0.08107993]
 [ 0.19380808  0.11593916  0.05382117]
 [-0.11289515 -0.624808    0.1055763 ]
 [ 0.52790358  0.72318188  0.04650274]]
```

#### Create a neural network model and check matrix dimensions

The network has two layers: layer 1 maps $M=4$ inputs to $D=5$ hidden units, and layer 2 maps $D=5$ hidden units to $K=3$ output classes.

In [9]:
n_inputs  = 4
n_hiddens = 5
n_classes = 3

l1 = myDenseLayer(n_hiddens, n_inputs)
l2 = myDenseLayer(n_classes, n_hiddens)

print(X_train.shape, y_train.shape)
print(l1.weight.shape, l1.bias.shape)
print(l2.weight.shape, l2.bias.shape)

(120, 4) (120, 3)
(5, 4) (5,)
(3, 5) (3,)


#### Backpropagation through activation functions

The softmax–cross-entropy combined gradient at the output layer (Eq. 3.23) is:

$$\frac{\partial J}{\partial \mathbf{z}^{[L]}} = \hat{\mathbf{y}} - \mathbf{y}$$

For hidden layers with sigmoid activation, the gradient is propagated using (Eq. 3.26):

$$\frac{\partial J}{\partial \mathbf{z}^{[l]}} = \left( (\mathbf{W}^{[l+1]})^T \frac{\partial J}{\partial \mathbf{z}^{[l+1]}} \right) \odot \sigma'(\mathbf{z}^{[l]})$$

where $\sigma'(\mathbf{z}^{[l]}) = \mathbf{a}^{[l]}(1 - \mathbf{a}^{[l]})$ is the sigmoid derivative (Section 3.5).

#### Define utility functions

In [10]:
from tensorflow.math import sigmoid as tf_sigmoid
from tensorflow.nn import softmax as tf_softmax

def sigmoid(x):
    x = tf_sigmoid(x)
    return x.numpy()

def softmax(x):
    x = tf_softmax(x)
    return x.numpy()

In [ ]:
def dJdz_sigmoid(wdJdz_upper, a_l):

    ### START CODE HERE ###
    
    # Apply sigmoid derivative: dJ/dz = (W^T * dJ/dz_upper) * a*(1-a)  (Eq. 3.26)
    dJdz = None
    
    ### END CODE HERE ###

    return dJdz

def dJdz_softmax(y_hat, y):

    ### START CODE HERE ###

    # Softmax-cross-entropy gradient: dJ/dz = y_hat - y  (Eq. 3.23)
    dJdz = None

    ### END CODE HERE ###
    
    return dJdz

In [12]:
np.random.seed(1)
c = dJdz_sigmoid(np.random.rand(2,3),np.random.rand(2,3))
d = dJdz_softmax(np.random.rand(2,3),np.random.rand(2,3))
print(c, '\n'+str(d))
c_exp = [[6.32069181e-02, 1.62900312e-01, 2.73748171e-05], [7.51276069e-02, 3.57307262e-02, 1.99168565e-02]]
d_exp = [[0.06406531, 0.68001595, -0.77335698], [-0.29779407, 0.10388062, -0.13363279]]
if not np.allclose(c, c_exp): print('dJdz_sigmoid failed')
elif not np.allclose(d, d_exp): print('dJdz_softmax failed')
else: print('functions passed')

[[6.32069181e-02 1.62900312e-01 2.73748171e-05]
 [7.51276069e-02 3.57307262e-02 1.99168565e-02]] 
[[ 0.06406531  0.68001595 -0.77335698]
 [-0.29779407  0.10388062 -0.13363279]]
functions passed


**Expected output:**
```
[[6.32069181e-02 1.62900312e-01 2.73748171e-05]
 [7.51276069e-02 3.57307262e-02 1.99168565e-02]]
[[ 0.06406531  0.68001595 -0.77335698]
 [-0.29779407  0.10388062 -0.13363279]]
```

#### Weight initialization (Section 3.9)

Proper weight initialization is critical for training (Section 3.9). Zero or identical-value initialization causes the weight symmetry problem, preventing learning. Random initialization breaks symmetry and enables gradient-based learning.

- Try different initializers: `np.zeros` / `np.ones` / `np.random.rand` / `np.random.randn`

In [ ]:
# Experiment with different initializers (Section 3.9)
np.random.seed(101)

### START CODE HERE ###

# Initialize weights with random values (symmetry breaking, Section 3.9)
l1.weight = None
l2.weight = None

### END CODE HERE ###

verify = True if np.allclose(l1.weight[0,0],2.706849839399938) else False

#### Training the two-layer neural network

The training loop implements the full backpropagation procedure (Section 3.7, Steps 1–7):
1. **Forward**: $\mathbf{a}^{[1]} = \sigma(\mathbf{W}^{[1]}\mathbf{x} + \mathbf{b}^{[1]})$, then $\hat{\mathbf{y}} = \text{softmax}(\mathbf{W}^{[2]}\mathbf{a}^{[1]} + \mathbf{b}^{[2]})$
2. **Backward**: Compute $\partial J/\partial \mathbf{z}^{[2]} = \hat{\mathbf{y}} - \mathbf{y}$ (Eq. 3.23), then propagate to layer 1 via Eq. 3.26
3. **Update**: $\mathbf{W}^{[l]} \leftarrow \mathbf{W}^{[l]} - \alpha \cdot \partial J/\partial \mathbf{W}^{[l]}$ (Eq. 3.28/3.30)

In [ ]:
# Hyperparameters
alpha = 0.1
n_epochs = 5000

for epoch in range(n_epochs):

    ### START CODE HERE ###

    # Forward: layer 1 with sigmoid activation (Section 3.6)
    a_1 = None
    # Forward: layer 2 with softmax activation (Eq. 3.13)
    a_2 = None

    # Backward: output layer gradient dJ/dz^[2] = y_hat - y (Eq. 3.23)
    dJdz_2 = None
    # Backward: layer 2 parameter gradients and propagate (Eq. 3.27)
    dw_2, db_2, wdJdz_2 = None
    # Backward: layer 1 gradient via sigmoid derivative (Eq. 3.26)
    dJdz_1 = None
    # Backward: layer 1 parameter gradients (Eq. 3.27)
    dw_1, db_1, _ = None

    # Update parameters via gradient descent (Eq. 3.28/3.30)
    l2.weight = None
    l2.bias = None
    l1.weight = None
    l1.bias = None

    # Compute cross-entropy loss (Eq. 3.16)
    loss_J = None

    ### END CODE HERE ###

    if ((epoch)%500==0):
        print('Epoch: %4d,  loss: %10.8f' % (epoch, loss_J))

print('Epoch: %4d,  loss: %10.8f' % (epoch, loss_J))

Epoch:    0,  loss: 1.16880164
Epoch:  500,  loss: 0.51621027
Epoch: 1000,  loss: 0.39778354
Epoch: 1500,  loss: 0.19370809
Epoch: 2000,  loss: 0.12675585
Epoch: 2500,  loss: 0.10084959
Epoch: 3000,  loss: 0.08774937
Epoch: 3500,  loss: 0.08001942
Epoch: 4000,  loss: 0.07495659
Epoch: 4500,  loss: 0.07137482
Epoch: 4999,  loss: 0.06869257


In [15]:
# Verify trained parameters against expected values
res = np.concatenate((l1.weight, l2.weight[:,0:4]), axis=0)
print(res)
exp = [[ 2.70684818,  0.62813159,  0.90796893,  0.50382567],
       [ 0.97535422,  1.25963926, -3.1792628,  -0.58561698],
       [-2.04873694,  0.71303734,  0.5321921,  -0.58526518],
       [ 1.93548842,  2.25935309, -3.04277186, -3.8387552 ],
       [ 0.19061251,  1.97863397,  2.60591617,  0.68350181],
       [-0.62486727,  7.99326304, -1.70023991,  0.15531964],
       [-0.05690593, -3.22128073,  0.1843417,   4.53523789],
       [ 2.01375351, -2.58170845, -0.50278989, -5.79604073]]
if verify:
    print('test passed.') if np.allclose(res, exp) else print('test failed.')
else: print('to verify your code, set initializer to '+'\033[1m'+'randn()'+'\033[0m')

[[ 2.70684818  0.62813159  0.90796893  0.50382567]
 [ 0.97535422  1.25963926 -3.1792628  -0.58561698]
 [-2.04873694  0.71303734  0.5321921  -0.58526518]
 [ 1.93548842  2.25935309 -3.04277186 -3.8387552 ]
 [ 0.19061251  1.97863397  2.60591617  0.68350181]
 [-0.62486727  7.99326304 -1.70023991  0.15531964]
 [-0.05690593 -3.22128073  0.1843417   4.53523789]
 [ 2.01375351 -2.58170845 -0.50278989 -5.79604073]]
test passed.


### Evaluate model performance

The trained model is evaluated on the test set by running forward propagation and comparing predicted classes to ground-truth labels.

In [ ]:
def my_predict(l1, l2, X_test):

    ### START CODE HERE ###

    # Forward: hidden layer with sigmoid
    y_hidd = None
    # Forward: output layer with softmax (Eq. 3.13)
    y_prod = None
    # Select class with highest probability (Eq. 3.4)
    y_pred = None

    ### END CODE HERE ###

    return y_pred

from sklearn.metrics import accuracy_score

y_pred = my_predict(l1, l2, X_test)

print(y_pred)
print(y_test)

accuracy_score(y_pred, y_test)

[0 0 0 2 1 2 1 1 2 0 2 0 0 2 2 1 1 1 0 2 1 0 1 1 1 1 1 2 0 0]
[0 0 0 2 1 2 1 1 2 0 2 0 0 2 2 1 1 1 0 2 1 0 1 1 1 1 1 2 0 0]


1.0

In [17]:
# Verify that predictions match ground truth on test set
if np.allclose(y_pred, y_test): print('test passed.')
else: print('test failed.')

test passed.


### Comparison with scikit-learn MLPClassifier

scikit-learn's `MLPClassifier` implements a multi-layer perceptron with configurable hidden layers, activation functions, and solvers. Here it uses a single hidden layer of 5 units with logistic (sigmoid) activation and SGD, comparable to the manual implementation above.

In [18]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(5,), activation='logistic', solver='sgd', \
                    alpha=0.01, learning_rate_init=0.1, max_iter=500)

# Training the model
mlp.fit(X_train, y_train_num)

# Making predictions
pred = mlp.predict(X_test)
accuracy_score(pred, y_test)

1.0

### Test model with a random sample

In [19]:
idx = np.random.randint(X_test.shape[0])
test_in = np.expand_dims(X_test[idx], axis=0)

species = ['setosa', 'versicolor', 'virginica']

y_pred = my_predict(l1, l2, test_in)
s_pred = mlp.predict(test_in)

print('My Prediction for Iris Species:', species[y_pred[0]])
print('SK Prediction for Iris Species:', species[s_pred[0]])
print('True Iris Species is:', species[y_test[idx]])

My Prediction for Iris Species: setosa
SK Prediction for Iris Species: setosa
True Iris Species is: setosa


(c) 2026 S. W. Lee